In [ ]:
!pip install pygeohash

In [ ]:
!pip install pygeohash catboost xgboost lightgbm joblib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 11.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import pygeohash as pgh
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import joblib

print("1. Loading Datasets...")
train = pd.read_csv('train.csv')
events = pd.read_csv('Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv')

print("2. Auto-Healing the Dataset Columns...")
print(f"Raw train.csv columns detected: {list(train.columns)}")

train.rename(columns={'hour': 'Hour', 'minute': 'Minute', 'road_type': 'RoadType', 'weather': 'Weather'}, inplace=True)

if 'timestamp' in train.columns and 'Hour' not in train.columns:
    train['Hour'] = train['timestamp'].str.split(':').str[0].astype(int)
    train['Minute'] = train['timestamp'].str.split(':').str[1].astype(int)

if 'Hour' not in train.columns:
    print("WARNING: Could not find time data! Automatically generating mock Hour/Minute to keep the AI pipeline alive.")
    train['Hour'] = np.random.randint(0, 24, size=len(train))
    train['Minute'] = np.random.choice([0, 15, 30, 45], size=len(train))

if 'RoadType' not in train.columns: train['RoadType'] = 'Unknown'
if 'Weather' not in train.columns: train['Weather'] = 'Unknown'

train['RoadType'] = train['RoadType'].fillna('Unknown')
train['Weather'] = train['Weather'].fillna('Unknown')

print("3. Converting GPS to Geohashes...")
events['geohash'] = events.apply(lambda row: pgh.encode(row['latitude'], row['longitude'], precision=6), axis=1)

print("4. Processing BTP Event Severities...")
severity_map = {
    'protest': 5, 'vip_movement': 5, 'public_event': 4, 'procession': 4,
    'water_logging': 3, 'accident': 3, 'construction': 2, 'tree_fall': 2,
    'vehicle_breakdown': 1, 'pot_holes': 1
}
events['Event_Scale'] = events['event_cause'].map(severity_map).fillna(1)

event_hotspots = events.groupby('geohash', as_index=False)['Event_Scale'].max()
event_hotspots.columns = ['geohash', 'Max_Historical_Event_Scale']

print("5. Merging and Feature Engineering (The V8 Logic)...")
train = train.merge(event_hotspots, on='geohash', how='left')
train['Max_Historical_Event_Scale'] = train['Max_Historical_Event_Scale'].fillna(0)
train['Active_Event_Scale'] = 0 # Default to 0 for training

# Spatial Features
train['geo_zone'] = train['geohash'].str[:5]
train['geo_district'] = train['geohash'].str[:4]
train['Road_Weather'] = train['RoadType'].astype(str) + "_" + train['Weather'].astype(str)

# Convert 'LargeVehicles' to numeric and add to categorical list
train['LargeVehicles'] = train['LargeVehicles'].map({'Allowed': 1, 'Not Allowed': 0}).fillna(0).astype(int)

# Convert 'Landmarks' to numeric and add to categorical list
train['Landmarks'] = train['Landmarks'].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)

# Cyclic Time Features
train['hour_sin'] = np.sin(2 * np.pi * train['Hour'] / 24.0)
train['hour_cos'] = np.cos(2 * np.pi * train['Hour'] / 24.0)
train['minute_sin'] = np.sin(2 * np.pi * train['Minute'] / 60.0)
train['minute_cos'] = np.cos(2 * np.pi * train['Minute'] / 60.0)

# Set Categoricals
cat_cols = ['geohash', 'RoadType', 'Weather', 'geo_zone', 'geo_district', 'Road_Weather', 'LargeVehicles', 'Landmarks']
for c in cat_cols:
    train[c] = train[c].astype('category')

# Prepare X and y
X = train.drop(columns=['demand', 'Index', 'day', 'timestamp'], errors='ignore')
y = train['demand']

print(f"Features ready for training: {list(X.columns)}")
print("\n6. Training the Production Holy Trinity Models (on 100% of data)...")


print("Training CatBoost (This takes about 60 seconds)...")
cb_model = CatBoostRegressor(
    iterations=1500, learning_rate=0.05, depth=8,
    cat_features=cat_cols, random_seed=42, verbose=0
)
cb_model.fit(X, y)


print("Training LightGBM...")
lgb_model = lgb.LGBMRegressor(
    n_estimators=1500, learning_rate=0.05, max_depth=8,
    random_state=42, n_jobs=-1
)
lgb_model.fit(X, y)

print("Training XGBoost...")
xgb_model = xgb.XGBRegressor(
    n_estimators=1500, learning_rate=0.05, max_depth=8,
    enable_categorical=True, tree_method='hist',
    random_state=42, n_jobs=-1
)
xgb_model.fit(X, y)

print("\n7. Exporting Models for the API...")
joblib.dump(cb_model, 'catboost_model.pkl')
joblib.dump(lgb_model, 'lightgbm_model.pkl')
joblib.dump(xgb_model, 'xgboost_model.pkl')

print("\nSUCCESS! All models trained and exported.")

1. Loading Datasets...
2. Auto-Healing the Dataset Columns...
Raw train.csv columns detected: ['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']
3. Converting GPS to Geohashes...
4. Processing BTP Event Severities...
5. Merging and Feature Engineering (The V8 Logic)...
Features ready for training: ['geohash', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'Hour', 'Minute', 'Max_Historical_Event_Scale', 'Active_Event_Scale', 'geo_zone', 'geo_district', 'Road_Weather', 'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos']

6. Training the Production Holy Trinity Models (on 100% of data)...
Training CatBoost (This takes about 60 seconds)...
Training LightGBM...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large num